# Hoofdstuk 7: Chatapplicaties bouwen
## Microsoft Foundry Models API Quickstart

Dit notitieboek is aangepast van de [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) die notitieboeken bevat die toegang hebben tot [Azure OpenAI](notebook-azure-openai.ipynb) services.

> **Opmerking:** GitHub Models wordt aan het einde van juli 2026 uitgefaseerd. Dit notitieboek richt zich nu op [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), dat dezelfde gratis te proberen modelcatalogus en Azure AI Inference SDK-ervaring biedt.


# Overzicht  
"Grote taalmodellen zijn functies die tekst naar tekst afbeelden. Gegeven een invoertekst probeert een groot taalmodel de tekst die daarna zal volgen te voorspellen"(1). Dit "snelstart" notitieboek zal gebruikers introduceren in hoog-niveau LLM-concepten, kernpakketvereisten om te beginnen met AML, een zachte introductie tot promptontwerp, en verschillende korte voorbeelden van verschillende gebruikssituaties. 


## Inhoudsopgave  

[Overzicht](#overview)  
[Hoe de OpenAI Service te gebruiken](#how-to-use-openai-service)  
[1. Je OpenAI Service aanmaken](#1.-creating-your-openai-service)  
[2. Installatie](#2.-installation)    
[3. Referenties](#3.-credentials)  

[Gebruikssituaties](#use-cases)    
[1. Tekst samenvatten](#1.-summarize-text)  
[2. Tekst classificeren](#2.-classify-text)  
[3. Nieuwe productnamen genereren](#3.-generate-new-product-names)  
[4. Een classifier fijn afstemmen](#4.fine-tune-a-classifier)  

[Referenties](#references)


### Bouw je eerste prompt  
Deze korte oefening geeft een basisintroductie voor het indienen van prompts bij een model in Microsoft Foundry Models voor een eenvoudige taak "samenvatting".


**Stappen**:  
1. Installeer de `azure-ai-inference` bibliotheek in je Python-omgeving, als je dat nog niet gedaan hebt.  
2. Laad standaard hulpprogramma-bibliotheken en stel je Microsoft Foundry Models-referenties in.  
3. Kies een model voor je taak  
4. Maak een eenvoudige prompt voor het model  
5. Dien je verzoek in bij de model-API!


### 1. Installeer `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importeer hulpbibliotheken en maak referenties aan


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Het juiste model vinden  
Modellen zoals GPT-4o en GPT-4o mini kunnen natuurlijke taal begrijpen en genereren, en zijn beschikbaar in de Microsoft Foundry Models-catalogus naast modellen van Meta, Mistral, Cohere en Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Promptontwerp  

"De magie van grote taalmodellen is dat ze door getraind te worden om deze voorspellingsfout te minimaliseren over enorme hoeveelheden tekst, uiteindelijk concepten leren die nuttig zijn voor deze voorspellingen. Ze leren bijvoorbeeld concepten zoals"(1):

* hoe je spelt
* hoe grammatica werkt
* hoe je parafraseert
* hoe je vragen beantwoordt
* hoe je een gesprek voert
* hoe je in veel talen schrijft
* hoe je codeert
* enzovoort.

#### Hoe een groot taalmodel te sturen  
"Van alle inputs aan een groot taalmodel is de tekstprompt veruit de meest invloedrijke(1).

Grote taalmodellen kunnen op een paar manieren worden gestimuleerd om output te produceren:

Instructie: Vertel het model wat je wilt
Voltooiing: Breng het model ertoe het begin van wat je wilt aan te vullen
Demonstratie: Laat het model zien wat je wilt, met ofwel:
Enkele voorbeelden in de prompt
Honderden of duizenden voorbeelden in een fijn-afgestelde trainingsdataset"



#### Er zijn drie basisrichtlijnen voor het maken van prompts:

**Laat zien en vertel**. Maak duidelijk wat je wilt via instructies, voorbeelden, of een combinatie van beide. Als je wilt dat het model een lijst items rangschikt op alfabetische volgorde of een alinea classificeert op sentiment, laat dan zien dat dit is wat je wilt.

**Lever kwaliteitsdata aan**. Als je probeert een classifier te bouwen of het model een patroon te laten volgen, zorg dan dat er genoeg voorbeelden zijn. Controleer je voorbeelden zorgvuldig—het model is meestal slim genoeg om basis spel- of typefouten te doorzien en je een antwoord te geven, maar het kan ook aannemen dat dit intentioneel is en het kan het antwoord beïnvloeden.

**Controleer je instellingen.** De temperatuur- en top_p-instellingen bepalen hoe deterministisch het model is bij het genereren van een antwoord. Als je een antwoord vraagt waarbij er maar één juist antwoord is, wil je deze instellingen lager zetten. Wil je meer diverse antwoorden, dan kun je ze hoger zetten. De grootste fout die mensen maken met deze instellingen is aannemen dat dit de "slimheid" of "creativiteit" regelt.


Bron: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Verzenden!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Dezelfde oproep herhalen, hoe vergelijken de resultaten? 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Tekst Samenvatten  
#### Uitdaging  
Vat tekst samen door een 'tl;dr:' toe te voegen aan het einde van een tekstgedeelte. Merk op hoe het model begrijpt hoe het een aantal taken moet uitvoeren zonder extra instructies. Je kunt experimenteren met meer beschrijvende prompts dan tl;dr om het gedrag van het model aan te passen en de samenvatting die je ontvangt te personaliseren(3).  

Recente onderzoeken hebben aanzienlijke verbeteringen laten zien op veel NLP-taken en benchmarks door voortraining op een grote tekstcorpus, gevolgd door fijn afstemmen op een specifieke taak. Hoewel de architectuur meestal taak-agnostisch is, vereist deze methode nog steeds taak-specifieke fijn afstemmingsdatasets van duizenden of tienduizenden voorbeelden. Daarentegen kunnen mensen over het algemeen een nieuwe taaltaak uitvoeren met slechts een paar voorbeelden of eenvoudige instructies - iets waar huidige NLP-systemen nog grotendeels moeite mee hebben. Hier tonen we aan dat het opschalen van taalmodellen de taak-agnostische, few-shot prestaties sterk verbetert, soms zelfs competitief met eerdere state-of-the-art fijn afstemmingsbenaderingen.  



Tl;dr


# Oefeningen voor verschillende gebruikssituaties  
1. Tekst samenvatten  
2. Tekst classificeren  
3. Nieuwe productnamen genereren


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Tekst classificeren  
#### Uitdaging  
Classificeer items in categorieën die tijdens de inferentie worden opgegeven. In het volgende voorbeeld geven we zowel de categorieën als de te classificeren tekst in de prompt (*playground_reference) op. 

Klantvraag: Hallo, een van de toetsen op mijn laptoptoetsenbord is onlangs gebroken en ik heb een vervanging nodig:

Geclassificeerde categorie:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Nieuwe Productnamen Genereren
#### Uitdaging
Maak productnamen aan vanuit voorbeeldwoorden. Hier voegen we in de prompt informatie toe over het product waarvoor we namen gaan genereren. We geven ook een vergelijkbaar voorbeeld om het patroon te tonen dat we willen ontvangen. We hebben ook de temperatuurwaarde hoog gezet om meer willekeur en innovatieve reacties te krijgen.

Productbeschrijving: Een thuis milkshakemaker
Zaadwoorden: snel, gezond, compact.
Productnamen: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Productbeschrijving: Een paar schoenen die op elke voetmaat passen.
Zaadwoorden: aanpasbaar, passend, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referenties  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Beste praktijken voor het fine-tunen van GPT-modellen om tekst te classificeren](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Voor Meer Hulp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bijdragers
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
